# Step 1: Train Maps (Playground)

In [ ]:
import pandas as pd
import numpy as np
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm
import os
import shutil
from math import radians, cos
import sys
sys.path.append(os.path.abspath(".."))

from data_lib.config import H_INSTALL, H_DECLINE, WEIGHT_DECLINE, WEIGHT_INSTALL, COMPETITION_SEARCH_RADIUS_DEG
import data_lib.config as config
from data_lib.data_fetch.get_data import get_train_data
from data_lib.feature.spatial_weights import build_desirability_field_idw
from data_lib.geometry.find_boundary import run_find_boundary
from data_lib.test import get_overlap


/Users/Rohanchoudhary/Desktop/projs/genie_stocks/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## ⚠️ Bug Fix: `data_lib/geometry/hex.py`

`install_velocity` line has `config.` leaked into an f-string:
```
row[f"installs_config.{config.TEMPORAL_WINDOWS[-1]}d"]   # ← produces 'installs_config_365d'
row[f"installs_{config.TEMPORAL_WINDOWS[-1]}d"]           # ← correct: 'installs_365d'
```

The cell below patches the source file directly so ProcessPoolExecutor workers pick it up.


In [2]:
import data_lib.geometry.hex as _hex_mod

hex_py_path = _hex_mod.__file__
print(f"Patching: {hex_py_path}")

with open(hex_py_path, "r") as f:
    src = f.read()

broken  = 'row[f"installs_config.{config.TEMPORAL_WINDOWS[-1]}d"]'
fixed   = 'row[f"installs_{config.TEMPORAL_WINDOWS[-1]}d"]'

if broken in src:
    src = src.replace(broken, fixed)
    with open(hex_py_path, "w") as f:
        f.write(src)
    print("✓ Fixed install_velocity bug in hex.py")
    
    # Force reimport so this process sees the fix too
    import importlib
    importlib.reload(_hex_mod)
    from data_lib.geometry.hex import compute_hexes
    print("✓ Module reloaded")
elif fixed in src:
    print("✓ Already fixed — nothing to do")
else:
    print("⚠️  Neither pattern found — check hex.py manually")


Patching: /Users/Rohanchoudhary/Desktop/projs/genie_stocks/data_lib/geometry/hex.py
✓ Already fixed — nothing to do


## 0. Setup

In [3]:
print("--- STARTING MAP TRAINING (PLAYGROUND) ---")

ARTIFACTS_DIR = "artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

print(f"Train window: {config.TRAIN_START_DATE} -> {config.TRAIN_END_DATE}")


--- STARTING MAP TRAINING (PLAYGROUND) ---
Train window: 2024-10-20 -> 2025-10-19


## 1. Get Data

In [4]:
df_train = get_train_data(config.TRAIN_START_DATE, config.TRAIN_END_DATE)
if df_train.empty:
    raise RuntimeError("No training data. Exiting.")
print(f"Training data: {len(df_train)} rows")
df_train.head()


Fetching TRAINING data (2024-10-20 to 2025-10-19)...
[WiomData] snowflake_select_start: select partner_id, mobile, first_notified_time, 
               case when lco_account_installed = pa
Standardization Cutoff (p40): 310.5 minutes
STANDARDISED DECISIONS (TRAIN):
final_decision
DECLINED         418973
INSTALLED        154324
INDETERMINATE     99067
HANGING           49358
Name: count, dtype: int64
Training data: 721722 rows


,partner_id,mobile,first_notified_time,installed_decision,latitude,longitude,installed_time,final_decision,active_base,partner_tenure,first_event_time,last_event,last_event_time,first_event,decision_time,time_to_decide
0,281749854726626,8860337457,2025-10-03 11:10:49,0,28.670757,77.276916,NaT,DECLINED,0,None,2025-10-03 11:11:16,DECLINED,2025-10-03 11:11:16,DECLINED,2025-10-03 11:11:16,0.000000
1,281749854623548,8512810790,2025-04-05 11:30:15,0,28.687747,77.262158,NaT,DECLINED,0,None,2025-04-05 11:50:28,DECLINED,2025-04-06 10:04:50,DECLINED,2025-04-05 11:50:28,1334.366667
2,281749854666636,9625602717,2025-07-27 12:18:44,0,28.504403,77.321651,2025-08-02 12:32:19,INDETERMINATE,0,None,2025-07-27 12:19:01,DECLINED,2025-08-01 07:00:11,INTERESTED,2025-07-27 12:19:01,6881.166667
3,274877915130,9058624387,2025-09-05 05:41:50,1,28.718234,77.266952,2025-09-05 08:23:54,INSTALLED,0,None,2025-09-05 05:42:01,Installation ongoing,2025-09-05 08:23:55,INTERESTED,2025-09-05 08:23:54,161.900000
4,281749854689559,7667911383,2024-11-13 08:04:06,0,28.635803,77.087570,2024-11-15 07:53:37,DECLINED,0,None,2024-11-13 16:13:02,DECLINED,2024-11-13 16:13:02,DECLINED,2024-11-13 16:13:02,0.000000


## 2. Build Desirability Fields

In [5]:
print("Building Desirability Fields...")
df_train = build_desirability_field_idw(
    df_train,
    radius_meters=max(H_INSTALL, H_DECLINE),
    decline_weight=WEIGHT_DECLINE,
    install_weight=WEIGHT_INSTALL,
)

# Set 'h' based on weight sign (Custom Logic for Split H)
df_train["h"] = np.where(df_train["field_weight"] >= 0, H_INSTALL, H_DECLINE)

# Save Train Data Artifact
train_h5_path = os.path.join(ARTIFACTS_DIR, "train_data.h5")
df_train.to_hdf(train_h5_path, mode="w", key="df")
print(f"Saved {train_h5_path}")


Building Desirability Fields...
Saved artifacts/train_data.h5


/var/folders/hx/vjf09zlx6_s9jfgrk3vz2d7w0000gp/T/ipykernel_40516/1729570263.py:14: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block8_values] [items->Index(['partner_tenure'], dtype='str')]

  df_train.to_hdf(train_h5_path, mode="w", key="df")


## 3. Calculate SE Thresholds

In [6]:
print("Generating Hexagons...")
partners = df_train["partner_id"].unique().tolist()

# Calculate SE thresholds
df_train["is_installed"] = (df_train["final_decision"] == "INSTALLED").astype(int)
df_train["is_declined"] = (df_train["final_decision"] == "DECLINED").astype(int)
df_train["is_indeterminate"] = (df_train["final_decision"] == "INDETERMINATE").astype(int)
df_train["is_hanging"] = (df_train["final_decision"] == "HANGING").astype(int)
df_train = df_train[df_train["final_decision"].isin(["INSTALLED", "DECLINED"])].copy()

dfp = df_train.groupby("partner_id").agg(
    total=("mobile", "count"), installed=("is_installed", "sum")
).reset_index()
dfp["se"] = dfp["installed"] / dfp["total"]
dfp["se_rng"] = pd.qcut(dfp["se"], q=10, labels=False, duplicates="drop") + 1

bad_se_rng = config.BAD_SE
mid_se_rng = config.MID_SE

bad_se = dfp[dfp["se_rng"].isin(bad_se_rng)]["se"].max()
mid_se = dfp[dfp["se_rng"].isin(mid_se_rng)]["se"].max()

print(f"bad_se = {bad_se:.4f}")
print(f"mid_se = {mid_se:.4f}")
print(f"Partners to process: {len(partners)}")


Generating Hexagons...
bad_se = 0.1450
mid_se = 0.4167
Partners to process: 1475


## 4. Hex Generation (Parallel)

In [7]:
from data_lib.geometry.hex import process_single_partner

hexagons = []
with ProcessPoolExecutor(max_workers=mp.cpu_count()) as executor:
    futures = {
        executor.submit(process_single_partner, pid, df_train, bad_se, mid_se): pid
        for pid in partners
    }
    for future in tqdm(as_completed(futures), total=len(futures), desc="Partners"):
        hexagons.extend(future.result())

# LET DICTS DEFINE SCHEMA — no hardcoded columns list
df_hex = pd.DataFrame(hexagons)

# Verify temporal columns made it through
temporal_check = [f"se_{wd}d" for wd in config.TEMPORAL_WINDOWS]
present = [c for c in temporal_check if c in df_hex.columns]
missing_t = [c for c in temporal_check if c not in df_hex.columns]
print(f"[HEX] Temporal columns present: {present}")
if missing_t:
    print(f"[HEX] WARNING: Temporal columns missing: {missing_t}")

# Verify install_velocity is clean
if "install_velocity" in df_hex.columns:
    n_valid = df_hex["install_velocity"].notna().sum()
    print(f"[HEX] install_velocity: {n_valid}/{len(df_hex)} valid")

# Check no ghost 'config' columns leaked in
bad_cols = [c for c in df_hex.columns if "config" in c.lower()]
if bad_cols:
    raise RuntimeError(f"Bug still present! Ghost columns: {bad_cols}")
else:
    print("[HEX] ✓ No 'config' ghost columns")

# Temporary save for find_boundary
temp_poly_path = "artifacts/poly_stats.h5"
df_hex.to_hdf(temp_poly_path, mode="w", key="df")
print(f"Saved intermediate {temp_poly_path} ({len(df_hex)} hexes, {len(df_hex.columns)} cols)")


Partners:   0%|          | 0/1475 [00:00<?, ?it/s]

MAX SEPARATION IS: 0.7394957983193278, BEST SIZE IS: 0.22


Partners:   0%|          | 1/1475 [00:08<3:21:54,  8.22s/it]

MAX SEPARATION IS: 0.6311688311688312, BEST SIZE IS: 0.05
MAX SEPARATION IS: 0.6764705882352942, BEST SIZE IS: 0.08


Partners:   0%|          | 3/1475 [00:11<1:14:56,  3.05s/it]

...

MAX SEPARATION IS: 0, BEST SIZE IS: 0.25


Partners: 100%|██████████| 1475/1475 [28:55<00:00,  1.18s/it]


[HEX] Temporal columns present: ['se_30d', 'se_60d', 'se_365d']
[HEX] install_velocity: 123493/123493 valid
[HEX] ✓ No 'config' ghost columns


/var/folders/hx/vjf09zlx6_s9jfgrk3vz2d7w0000gp/T/ipykernel_40516/571620601.py:37: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block4_values] [items->Index(['poly'], dtype='str')]

  df_hex.to_hdf(temp_poly_path, mode="w", key="df")


Saved intermediate artifacts/poly_stats.h5 (123493 hexes, 22 cols)


## 5. Find Boundaries

In [8]:
print("Finding Boundaries...")
run_find_boundary()

# Move result to artifacts if it ended up in cwd
bound_path = "partner_cluster_boundaries.h5"
artifact_bound_path = os.path.join(ARTIFACTS_DIR, "partner_cluster_boundaries.h5")

if os.path.exists(bound_path) and os.path.abspath(bound_path) != os.path.abspath(artifact_bound_path):
    shutil.move(bound_path, artifact_bound_path)
    print(f"Moved {bound_path} → {artifact_bound_path}")
else:
    print(f"Boundary file at {artifact_bound_path}")


Finding Boundaries...
IDENTIFYING HEX CENTROID LAT LONG
FILTERING SERVICEABLE HEXES
count    37736.000000
mean         3.926675
std          8.561481
min          1.000000
25%          1.000000
50%          2.000000
75%          3.000000
max        252.000000
Name: installs, dtype: float64
p50 value is 4, so filtering for hexagons > 4
COMPUTING INSTALLS QUANTILES
    quantile  value
0        0.0    1.0
1        0.1    1.0
2        0.2    1.0
3        0.3    1.0
4        0.4    1.0
5        0.5    2.0
6        0.6    2.0
7        0.7    3.0
8        0.8    4.0
9        0.9    8.0
10       1.0  252.0
Dynamic p30: 1.0 | p70: 3.0 | p90: 8.0
Filtering hexagons with installs > p30 (1.0)


CLUSTERING_DBSCAN:   0%|          | 1/1260 [00:00<02:30,  8.39it/s]

Partner 274877909269: Rescuing 1 high-value singleton(s) (>= p90 installs)
Partner 274877918547: Rescuing 1 high-value singleton(s) (>= p90 installs)


CLUSTERING_DBSCAN:  17%|█▋        | 211/1260 [00:00<00:01, 548.85it/s]

Partner 274877928382: Rescuing 1 high-value singleton(s) (>= p90 installs)
Partner 274877931670: Rescuing 1 high-value singleton(s) (>= p90 installs)
Partner 274877932288: Rescuing 1 high-value singleton(s) (>= p90 installs)
Partner 274877936145: Rescuing 1 high-value singleton(s) (>= p90 installs)
Partner 274877938294: Rescuing 1 high-value singleton(s) (>= p90 installs)
Partner 274877942891: Rescuing 1 high-value singleton(s) (>= p90 installs)
Partner 281749854617958: Rescuing 1 high-value singleton(s) (>= p90 installs)
Partner 281749854619437: Rescuing 1 high-value singleton(s) (>= p90 installs)
Partner 281749854623773: Rescuing 1 high-value singleton(s) (>= p90 installs)


...

Generated map for Partner 281749854808655: 'artifacts/virtual_boundary/partner_281749854808655_clusters.html'
Generated map for Partner 281749854808980: 'artifacts/virtual_boundary/partner_281749854808980_clusters.html'
Generated map for Partner 281749854808981: 'artifacts/virtual_boundary/partner_281749854808981_clusters.html'
Generated map for Partner 281749854808982: 'artifacts/virtual_boundary/partner_281749854808982_clusters.html'
Generated map for Partner 281749854808983: 'artifacts/virtual_boundary/partner_281749854808983_clusters.html'
Generated map for Partner 281749854808984: 'artifacts/virtual_boundary/partner_281749854808984_clusters.html'
Generated map for Partner 281749854809233: 'artifacts/virtual_boundary/partner_281749854809233_clusters.html'
Generated map for Partner 281749854809462: 'artifacts/virtual_boundary/partner_281749854809462_clusters.html'
Generated map for Partner 281749854809463: 'artifacts/virtual_boundary/partner_281749854809463_clusters.html'
Generated 

PARTNER_MAPS:  98%|█████████▊| 1126/1144 [00:05<00:00, 229.18it/s]

Generated map for Partner 281749854815209: 'artifacts/virtual_boundary/partner_281749854815209_clusters.html'
Generated map for Partner 281749854816386: 'artifacts/virtual_boundary/partner_281749854816386_clusters.html'
Generated map for Partner 281749854816387: 'artifacts/virtual_boundary/partner_281749854816387_clusters.html'
Generated map for Partner 281749854816388: 'artifacts/virtual_boundary/partner_281749854816388_clusters.html'
Generated map for Partner 281749854817595: 'artifacts/virtual_boundary/partner_281749854817595_clusters.html'
Generated map for Partner 281749854819228: 'artifacts/virtual_boundary/partner_281749854819228_clusters.html'
Generated map for Partner 281749854819229: 'artifacts/virtual_boundary/partner_281749854819229_clusters.html'
Generated map for Partner 281749854819402: 'artifacts/virtual_boundary/partner_281749854819402_clusters.html'
Generated map for Partner 281749854819418: 'artifacts/virtual_boundary/partner_281749854819418_clusters.html'
Generated 

PARTNER_MAPS: 100%|██████████| 1144/1144 [00:05<00:00, 212.12it/s]

Generated map for Partner 281749854851123: 'artifacts/virtual_boundary/partner_281749854851123_clusters.html'
Generated map for Partner 281749854852201: 'artifacts/virtual_boundary/partner_281749854852201_clusters.html'
Generated map for Partner 281749854855727: 'artifacts/virtual_boundary/partner_281749854855727_clusters.html'
Generated map for Partner 281749854855765: 'artifacts/virtual_boundary/partner_281749854855765_clusters.html'
Generated map for Partner 281749854856817: 'artifacts/virtual_boundary/partner_281749854856817_clusters.html'
Generated map for Partner 281749854859134: 'artifacts/virtual_boundary/partner_281749854859134_clusters.html'
Generated map for Partner 281749854859135: 'artifacts/virtual_boundary/partner_281749854859135_clusters.html'
Generated map for Partner 281749854859136: 'artifacts/virtual_boundary/partner_281749854859136_clusters.html'
Generated map for Partner 281749854860114: 'artifacts/virtual_boundary/partner_281749854860114_clusters.html'
Generated 

## 6. Competition / Overlaps (Final Map)

In [9]:
print("Computing Competition Overlaps...")

# get_overlap() reads from artifacts/ directly, but also needs
# partner_cluster_boundaries.h5 accessible in cwd — copy if needed
bound_path = "partner_cluster_boundaries.h5"
artifact_bound_path = os.path.join(ARTIFACTS_DIR, "partner_cluster_boundaries.h5")

if os.path.exists(artifact_bound_path) and not os.path.exists(bound_path):
    shutil.copy(artifact_bound_path, bound_path)

get_overlap(
    search_radius_deg=COMPETITION_SEARCH_RADIUS_DEG
)

# Move final poly output to artifacts
final_poly_path = "poly_stats_final.h5"
artifact_final_poly_path = os.path.join(ARTIFACTS_DIR, "poly_stats_final.h5")

if os.path.exists(final_poly_path) and os.path.abspath(final_poly_path) != os.path.abspath(artifact_final_poly_path):
    shutil.move(final_poly_path, artifact_final_poly_path)
    print(f"Saved {artifact_final_poly_path}")
else:
    print(f"Final poly file at {artifact_final_poly_path}")

print("--- MAP TRAINING COMPLETE ---")


Computing Competition Overlaps...
Loading data...
Loaded 123,493 hexagons and 2,376 cluster boundaries.

Computing distances to ALL cluster boundaries (own + foreign)...


Distance to nearest clusters: 100%|██████████| 123493/123493 [00:36<00:00, 3380.10it/s]
/Users/Rohanchoudhary/Desktop/projs/genie_stocks/data_lib/test.py:96: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block3_values] [items->Index(['partner_id', 'poly', 'color'], dtype='str')]

  df_hex.to_hdf(


Distance computation complete:
   → 85,233 hexes overlap foreign territory
   → 114,483 hexes have own territory nearby

Ranking crimson hexes by proximity to own territory...
Ranked 85,757 crimson hexes.

Finalizing dataset...


/Users/Rohanchoudhary/Desktop/projs/genie_stocks/data_lib/test.py:159: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block3_values] [items->Index(['partner_id', 'poly', 'color'], dtype='str')]

  df_final.to_hdf(


Final poly file at artifacts/poly_stats_final.h5
--- MAP TRAINING COMPLETE ---


## 7. Verify Outputs

In [10]:
expected_files = [
    "artifacts/train_data.h5",
    "artifacts/poly_stats.h5",
    "artifacts/partner_cluster_boundaries.h5",
    "artifacts/poly_stats_final.h5",
]

print("Output files:")
for f in expected_files:
    exists = os.path.exists(f)
    size = os.path.getsize(f) / 1024 / 1024 if exists else 0
    status = f"✓ ({size:.1f} MB)" if exists else "✗ MISSING"
    print(f"  {f}: {status}")

# Quick column check on final poly
df_final_check = pd.read_hdf("artifacts/poly_stats_final.h5", "df")
print(f"\nFinal poly: {len(df_final_check)} rows, {len(df_final_check.columns)} cols")
print(f"Columns: {sorted(df_final_check.columns.tolist())}")

# Confirm no 'installs_config_*' ghost columns
bad_cols = [c for c in df_final_check.columns if "config" in c.lower()]
if bad_cols:
    print(f"\n⚠️ WARNING: Found 'config' in column names (bug residue): {bad_cols}")
else:
    print("\n✓ No 'config' ghost columns — bug fix confirmed clean")


Output files:
  artifacts/train_data.h5: ✓ (125.1 MB)
  artifacts/poly_stats.h5: ✓ (38.7 MB)
  artifacts/partner_cluster_boundaries.h5: ✓ (15.7 MB)
  artifacts/poly_stats_final.h5: ✓ (21.9 MB)

Final poly: 123493 rows, 26 cols
Columns: ['best_size', 'color', 'declines', 'declines_30d', 'declines_365d', 'declines_60d', 'distance_from_boundary_m', 'distance_to_own_boundary_m', 'install_velocity', 'installs', 'installs_30d', 'installs_365d', 'installs_60d', 'is_overlap', 'partner_id', 'poly', 'poly_id', 'rank', 'se', 'se_30d', 'se_365d', 'se_60d', 'total', 'total_30d', 'total_365d', 'total_60d']

✓ No 'config' ghost columns — bug fix confirmed clean
